# Grammar Error Detection — Word-Level (Which Word Is Wrong?)

This version labels **every individual word** as CORRECT or INCORRECT, using one token-classification model (the same idea BERT uses for Named Entity Recognition — but here "the entity" is a grammar mistake).

**Why one model is enough for both questions:**
- If **any** word in a sentence is predicted INCORRECT → the sentence is INCORRECT.
- The flagged word(s) tell you exactly *where* the mistake is.

No separate sentence-level model needed.

### Step 1: Load the data

In [1]:
import pandas as pd

df = pd.read_json(r"C:\Users\Adini\Code_files\GEN AI\NLP\Grammar Error Detection\Dataset\fce.train.json", lines=True)
df.head()

,text,age,q,script-s,edits,l1,id,answer-s
0,"Dear Sir or Madam,\n\nI am writing in order to...",21-25,1,31,"[[0, [[71, 76, 'with', 'RT'], [118, 122, 'saw'...",ca,TR1*0102*2000*01,4.3
1,"Unfortunately, Pat wasn't very good at keeping...",21-25,2,31,"[[0, [[233, 233, ',', 'MP'], [249, 249, '""', '...",ca,TR1*0102*2000*01,5.1
2,"10 June 2000\n\nDear Manager,\n\nI would like ...",26-30,1,29,"[[0, [[238, 238, 'in', 'MT'], [473, 473, 'the'...",ko,TR2*0102*2000*01,3.3
3,Fashion of the future\n\nPeople will wear this...,26-30,3,29,"[[0, [[71, 76, '', 'UJ'], [91, 91, 'the', 'MD'...",ko,TR2*0102*2000*01,3.3
4,DECEMBER 12TH\n\nPRINCIPAL MR. ROBERTSON\n\nDE...,16-20,1,28,"[[0, [[61, 65, 'THANK', 'S'], [142, 144, 'ON',...",ca,TR3*0100*2000*02,2.3


### Step 2: Split essays into sentences, then label every word

1. Split each essay into sentences (keeping character positions).
2. Pull out the (start, end) spans where real edits happened.
3. Split each sentence into words (also keeping character positions).
4. Label a **word** `1` (incorrect) only if an edit span overlaps that specific word — `0` otherwise.
5. A sentence's overall label is simply: incorrect if **any** of its words are incorrect.

In [2]:
import re

def split_into_sentences_with_spans(text):
    """Sentence splitter that keeps each sentence's exact (start, end) position
    in the original essay (leading/trailing whitespace stripped from the span too)."""
    sentences, spans = [], []
    for match in re.finditer(r"[^.!?\n]+[.!?]?", text):
        raw = match.group()
        sent = raw.strip()
        if sent:
            lstrip_len = len(raw) - len(raw.lstrip())
            start = match.start() + lstrip_len
            end = start + len(sent)
            sentences.append(sent)
            spans.append((start, end))
    return sentences, spans


def get_error_spans(edits_raw):
    """Pull out (start, end) character spans from the edits column."""
    if not edits_raw:
        return []
    annotator_edits = edits_raw[0][1]   # edits = [[annotator_id, [edit, edit, ...]]]
    return [(edit[0], edit[1]) for edit in annotator_edits]


def overlaps(a_start, a_end, b_start, b_end):
    """True if two character spans touch or overlap."""
    if b_start == b_end:              # zero-width edit = a missing word inserted here
        return a_start < b_start < a_end
    return b_start < a_end and b_end > a_start


def label_words(sentence, sent_start, error_spans):
    """Split a sentence into words and label each one 0 (correct) / 1 (incorrect)."""
    words, tags = [], []
    for match in re.finditer(r"\S+", sentence):
        w_start = sent_start + match.start()
        w_end = sent_start + match.end()
        is_incorrect = any(overlaps(w_start, w_end, e_start, e_end) for e_start, e_end in error_spans)
        words.append(match.group())
        tags.append(int(is_incorrect))
    return words, tags


rows = []
for _, row in df.iterrows():
    sentences, spans = split_into_sentences_with_spans(row["text"])
    error_spans = get_error_spans(row["edits"])
    for sent, (s_start, s_end) in zip(sentences, spans):
        if len(sent.split()) < 3:          # skip junk fragments (headers, single words, etc.)
            continue
        words, tags = label_words(sent, s_start, error_spans)
        rows.append({"text": sent, "words": words, "tags": tags, "label": int(any(tags))})

word_df = pd.DataFrame(rows).drop_duplicates(subset="text").reset_index(drop=True)
word_df.head()

,text,words,tags,label
0,"Dear Sir or Madam,","[Dear, Sir, or, Madam,]","[0, 0, 0, 0]",0
1,I am writing in order to express my disappoint...,"[I, am, writing, in, order, to, express, my, d...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]",1
2,I saws the show's advertisement hanging up of ...,"[I, saws, the, show's, advertisement, hanging,...","[0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...",1
3,I convinced them to go there with me because I...,"[I, convinced, them, to, go, there, with, me, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, ...",1
4,"The problems started in the box office, where ...","[The, problems, started, in, the, box, office,...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1


In [3]:
word_df["label"].value_counts()

label
1    16834
0     8762
Name: count, dtype: int64

In [4]:
# word-level balance — this is what the token classifier actually learns from
total_words = sum(len(t) for t in word_df["tags"])
incorrect_words = sum(sum(t) for t in word_df["tags"])
print(f"Total words: {total_words}")
print(f"Incorrect words: {incorrect_words} ({incorrect_words/total_words:.1%})")

Total words: 389696
Incorrect words: 50253 (12.9%)


### Step 3: Split into train and test

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    word_df, test_size=0.15, random_state=42, stratify=word_df["label"]
)

train_df.shape, test_df.shape

((21756, 4), (3840, 4))

### Step 4: Load a pretrained BERT tokenizer and *token classification* model

`AutoModelForSequenceClassification`: It outputs one CORRECT/INCORRECT prediction **per word**, not one for the whole sentence.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_NAME = "distilbert-base-uncased"

id2label = {0: "CORRECT", 1: "INCORRECT"}
label2id = {"CORRECT": 0, "INCORRECT": 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)

print(torch.cuda.is_available())

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


False


### Step 5: Put the train/test data into a simple table BERT can read

In [7]:
from datasets import Dataset

train_data = Dataset.from_dict({"tokens": train_df["words"].tolist(), "tags": train_df["tags"].tolist()})
test_data = Dataset.from_dict({"tokens": test_df["words"].tolist(), "tags": test_df["tags"].tolist()})

### Step 6: Tokenize words into subwords, and line up labels with them

BERT splits some words into multiple subword pieces (e.g. "unhappiness" → "un", "##happiness"). We only put the real label on a word's **first** subword piece, and mark the rest (and special tokens like `[CLS]`) with `-100`, which tells the model "don't score this position".

In [8]:
def tokenize_and_align_labels(batch):
    tokenized = tokenizer(
        batch["tokens"], truncation=True, is_split_into_words=True,
        padding="max_length", max_length=32
    )
    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)                    # [CLS], [SEP], padding
            elif word_idx != previous_word_idx:
                label_ids.append(tags[word_idx])           # first subword of a word
            else:
                label_ids.append(-100)                    # later subwords of the same word
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized


train_data = train_data.map(tokenize_and_align_labels, batched=True)
test_data = test_data.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/21756 [00:00<?, ? examples/s]

Map:   0%|          | 0/3840 [00:00<?, ? examples/s]

### Step 7: Train the model (only a few epochs — it's already pretrained)

In [9]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="bert_ged_token_ckpt",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    eval_strategy="no",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

trainer.train()

C:\Users\Adini\anaconda3\envs\Bhuvana\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,0.346777
100,0.317021
150,0.290162
200,0.286472
250,0.289564
300,0.300912
350,0.284344
400,0.284675
450,0.272123
500,0.274464


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=680, training_loss=0.2903951644897461, metrics={'train_runtime': 3298.9805, 'train_samples_per_second': 6.595, 'train_steps_per_second': 0.206, 'total_flos': 177655552768512.0, 'train_loss': 0.2903951644897461, 'epoch': 1.0})

### Step 8: Check word-level accuracy on the test set

We only score positions that aren't `-100` — i.e. real words, not padding or subword continuations.

In [10]:
import numpy as np
from sklearn.metrics import classification_report

raw_preds = trainer.predict(test_data)
preds = np.argmax(raw_preds.predictions, axis=-1)
labels = raw_preds.label_ids

true_labels, true_preds = [], []
for pred_row, label_row in zip(preds, labels):
    for p, l in zip(pred_row, label_row):
        if l != -100:
            true_labels.append(l)
            true_preds.append(p)

print(
    classification_report(
        true_labels, true_preds,
        labels=[0, 1], target_names=["CORRECT", "INCORRECT"], zero_division=0,
    )
)

C:\Users\Adini\anaconda3\envs\Bhuvana\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

     CORRECT       0.90      0.99      0.94     48725
   INCORRECT       0.79      0.28      0.42      7174

    accuracy                           0.90     55899
   macro avg       0.84      0.64      0.68     55899
weighted avg       0.89      0.90      0.88     55899



### Step 9: Try it on your own sentences — now with the flagged word(s)

In [11]:
def find_grammar_errors(text):
    words = text.split()
    tokens = tokenizer(words, return_tensors="pt", is_split_into_words=True, truncation=True)

    with torch.no_grad():
        output = model(**tokens)
    predicted_ids = output.logits.argmax(dim=-1)[0].tolist()
    word_ids = tokens.word_ids(batch_index=0)

    # keep only the first subword's prediction for each word
    word_labels = {}
    for word_idx, pred_id in zip(word_ids, predicted_ids):
        if word_idx is not None and word_idx not in word_labels:
            word_labels[word_idx] = id2label[pred_id]

    flagged_words = [words[i] for i, label in word_labels.items() if label == "INCORRECT"]

    if flagged_words:
        return "INCORRECT", flagged_words
    return "CORRECT", []


find_grammar_errors("She go to school every day")

('INCORRECT', ['go'])

In [12]:
find_grammar_errors("She goes to school every day")

('CORRECT', [])

In [13]:
find_grammar_errors("I have went to the store yesterday")

('CORRECT', [])

In [14]:
find_grammar_errors("He doesn't like coffee")

('CORRECT', [])

In [15]:
find_grammar_errors("Yesterday I go to the market and buyed some fruits")

('INCORRECT', ['buyed'])

### Step 10: Save the model for later use

In [16]:
model.save_pretrained("distilbert_ged_token_model1")
tokenizer.save_pretrained("distilbert_ged_token_model1")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert_ged_token_model1\\tokenizer_config.json',
 'distilbert_ged_token_model1\\tokenizer.json')

In [5]:
import numpy as np
arr = np.array([0,1,1,0,1])
np.argmax(arr)

np.int64(1)